# Vectorized Grid Search Analysis

This notebook analyzes outputs from `run_2d_grid_search_vectorized.py` / `run_2d_grid_search_vectorized.sh`.

It expects merged vectorized results in `grid_search_results_vec.npz`.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Point to vectorized output directory (override as needed)
RESULTS_DIR = '/om2/user/bjmedina/auditory-memory/memory/reports/figures/2d_grid_search'
VEC_PATH = os.path.join(RESULTS_DIR, 'grid_search_results_vec.npz')

vec = np.load(VEC_PATH)
sigma0_grid = vec['sigma0_grid']
sigma_grid = vec['sigma_grid']
eta_grid = vec['eta_grid']
ISI_VALUES = tuple(vec['isi_values'].astype(int))

results = {isi: vec[f'dprime_isi{isi}'] for isi in ISI_VALUES}

print(f'Loaded: {VEC_PATH}')
print(f'Grid shape: {len(sigma0_grid)} x {len(sigma_grid)} x {len(eta_grid)}')
print(f'ISI values: {ISI_VALUES}')


## Build tidy dataframe and basic QC


In [ ]:
rows = []
for i_s0, s0 in enumerate(sigma0_grid):
    for i_sig, sig in enumerate(sigma_grid):
        for i_eta, eta in enumerate(eta_grid):
            row = {
                'sigma0': float(s0),
                'sigma': float(sig),
                'eta': float(eta),
                'i_sigma0': i_s0,
                'i_sigma': i_sig,
                'i_eta': i_eta,
            }
            for isi in ISI_VALUES:
                row[f'dprime_isi{isi}'] = float(results[isi][i_s0, i_sig, i_eta])
            rows.append(row)

df = pd.DataFrame(rows)

valid_mask = np.ones(len(df), dtype=bool)
for isi in ISI_VALUES:
    valid_mask &= np.isfinite(df[f'dprime_isi{isi}'])

df_valid = df[valid_mask].copy()
print(f'Total grid points: {len(df)}')
print(f'Complete points (all ISIs finite): {len(df_valid)}')
print(f'Missing points: {len(df) - len(df_valid)}')
display(df_valid.head())


## d' distributions per ISI (vectorized run)


In [ ]:
fig, axes = plt.subplots(1, len(ISI_VALUES), figsize=(4.5 * len(ISI_VALUES), 3.8), sharey=True)
axes = np.atleast_1d(axes)

for i, isi in enumerate(ISI_VALUES):
    ax = axes[i]
    vals = df_valid[f'dprime_isi{isi}'].values
    ax.hist(vals, bins=30, alpha=0.8, color='steelblue')
    ax.set_title(f'ISI={isi}')
    ax.set_xlabel("d'")
    if i == 0:
        ax.set_ylabel('Count')

fig.suptitle("Distribution of d' across grid points", y=1.03)
fig.tight_layout()
plt.show()


## Human-like decay score (vectorized-only)

This mirrors the ranking idea used in the non-vectorized notebook while staying compatible with vectorized outputs:

- `delta_total = d'(ISI0) - d'(ISI16)`
- `delta_early = d'(ISI0) - d'(ISI2)` (when ISI=2 exists)
- `delta_late = d'(ISI8) - d'(ISI16)` (when ISI=8 exists; otherwise defaults to 0)
- penalty for extreme ceiling/floor trajectories


In [ ]:
def zscore(s):
    std = s.std(ddof=0)
    if std == 0 or np.isnan(std):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

work = df_valid.copy()

if 0 in ISI_VALUES and 16 in ISI_VALUES:
    work['delta_total'] = work['dprime_isi0'] - work['dprime_isi16']
else:
    raise ValueError(f'Need ISI=0 and ISI=16 for decay ranking, found {ISI_VALUES}')

if 2 in ISI_VALUES:
    work['delta_early'] = work['dprime_isi0'] - work['dprime_isi2']
else:
    work['delta_early'] = 0.0

if 8 in ISI_VALUES:
    work['delta_late'] = work['dprime_isi8'] - work['dprime_isi16']
else:
    work['delta_late'] = 0.0

work['penalty_extreme'] = np.maximum(work['dprime_isi0'] - 3.0, 0) + np.maximum(0.3 - work['dprime_isi16'], 0)

work['decay_score'] = (
    1.0 * zscore(work['delta_total']) +
    0.8 * zscore(work['delta_early']) -
    0.8 * zscore(np.abs(work['delta_late'])) -
    1.2 * zscore(work['penalty_extreme'])
)

interesting = work[(work['dprime_isi0'] > 0.4) & (work['delta_total'] > 0.1)].copy()
top = interesting.nlargest(15, 'decay_score').reset_index(drop=True)

display(top[[
    'sigma0', 'sigma', 'eta',
    'dprime_isi0', 'dprime_isi2' if 2 in ISI_VALUES else 'dprime_isi0',
    'dprime_isi16',
    'delta_total', 'delta_early', 'delta_late', 'penalty_extreme', 'decay_score'
]].head(15))


## ISI curves for top-ranked vectorized triples


In [ ]:
isi_arr = np.array(ISI_VALUES)
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.tab20(np.linspace(0, 1, min(12, len(top))))

for i, (_, row) in enumerate(top.head(12).iterrows()):
    curve = [row[f'dprime_isi{isi}'] for isi in ISI_VALUES]
    ax.plot(isi_arr, curve, marker='o', linewidth=1.8, color=colors[i],
            label=f"σ₀={row['sigma0']:.2f}, σ={row['sigma']:.3f}, η={row['eta']:.3f}")

ax.set_xlabel('ISI')
ax.set_ylabel("d'")
ax.set_title('Top vectorized triples by human-like decay score')
ax.set_xticks(isi_arr)
ax.legend(fontsize=7, ncol=2, loc='best')
fig.tight_layout()
plt.show()


## Tradeoff scatter: total decay vs late-plateau change


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    work['delta_total'],
    np.abs(work['delta_late']),
    c=work['dprime_isi0'],
    cmap='viridis',
    alpha=0.7,
    s=35,
)
ax.set_xlabel("Total decay: d'(ISI0) - d'(ISI16)")
ax.set_ylabel("Late change magnitude: |d'(ISI8)-d'(ISI16)|")
ax.set_title('Prefer upper-left? no; prefer high total decay + small late change')
cb = plt.colorbar(sc, ax=ax)
cb.set_label("d'(ISI0)")
fig.tight_layout()
plt.show()


## Export candidate triples for single-ISI follow-up jobs


In [ ]:
out_csv = os.path.join(RESULTS_DIR, 'vectorized_top_triples_for_followup.csv')
cols = ['sigma0', 'sigma', 'eta', 'decay_score'] + [f'dprime_isi{isi}' for isi in ISI_VALUES]
top[cols].to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')

print('Example next step: run ISI=16-only follow-up for a selected triple set in a dedicated script.')
